## TOKENIZATION


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [5]:
df = pd.read_csv("../data/processed/processed_imdb.csv")
df.head()

,review,sentiment,label,clean_review,processed_review,word_count,processed_word_count
0,One of the other reviewers has mentioned that ...,positive,1,one of the other reviewers has mentioned that ...,reviewer mention watch oz episode ll hook righ...,307,154
1,A wonderful little production. <br /><br />The...,positive,1,a wonderful little production the filming tech...,wonderful little production filming technique ...,162,80
2,I thought this was a wonderful way to spend ti...,positive,1,i thought this was a wonderful way to spend ti...,think wonderful way spend time hot summer week...,166,78
3,Basically there's a family where a little boy ...,negative,0,basically there s a family where a little boy ...,basically s family little boy jake think s zom...,138,60
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1,petter mattei s love in the time of money is a...,petter mattei s love time money visually stunn...,230,115


In [6]:
print(df.columns)
print(df[["processed_review","label"]])

Index(['review', 'sentiment', 'label', 'clean_review', 'processed_review',
       'word_count', 'processed_word_count'],
      dtype='str')
                                        processed_review  label
0      reviewer mention watch oz episode ll hook righ...      1
1      wonderful little production filming technique ...      1
2      think wonderful way spend time hot summer week...      1
3      basically s family little boy jake think s zom...      0
4      petter mattei s love time money visually stunn...      1
...                                                  ...    ...
49995  think movie right good job wasn t creative ori...      1
49996  bad plot bad dialogue bad acting idiotic direc...      0
49997  catholic teach parochial elementary school nun...      0
49998  m go disagree previous comment maltin second r...      0
49999  no expect star trek movie high art fan expect ...      0

[50000 rows x 2 columns]


In [7]:
X =df["processed_review"]
y=df["label"]


In [10]:
X_train,X_test,y_train,y_test =train_test_split(
    X,
    y,
    test_size = 0.15,
    random_state =42,
    stratify = y
)

In [45]:
X_train,X_val,y_train,y_val = train_test_split(
    X_train,
    y_train,
    test_size = 0.1765, # %15 valid
    random_state=42,
    stratify=y_train
)

In [12]:
print("Train size:",len(X_train))
print("Validation size:",len(X_val))
print("Test size:",len(X_test))

Train size: 34998
Validation size: 7502
Test size: 7500


In [15]:
print(y_train.value_counts(normalize=True))
print(y_val.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))


label
0    0.5
1    0.5
Name: proportion, dtype: float64
label
1    0.5
0    0.5
Name: proportion, dtype: float64
label
0    0.5
1    0.5
Name: proportion, dtype: float64


In [16]:
VOCAB_SIZE = 10000

OOV_TOKEN = "<OOV>"

In [17]:
tokenizer = Tokenizer(
    num_words = VOCAB_SIZE,
    oov_token = OOV_TOKEN
)

In [ ]:
tokenizer.fit_on_texts(X_train)

None


In [19]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [22]:
print("Orginal text:\n",X_train.iloc[0])
print("\nSequence:\n",X_train_seq[0])

Orginal text:
 love bad old skifee movie people understand budget budget say planet dinosaur bad bad movie thing no actor attractive female kill minute swim ashore literally no redeem quality find pile waste celluloid thing not waste paper screenplay no page long surely no actually write dialogue pointless m constantly amazed movie get release m glad didn t pay waste time s minute life ll never

Sequence:
 [25, 12, 51, 1, 3, 21, 193, 184, 184, 64, 962, 1922, 12, 12, 3, 27, 13, 36, 1252, 427, 87, 79, 2946, 1, 1008, 13, 1062, 286, 30, 1768, 166, 2974, 27, 6, 166, 1490, 674, 13, 1233, 73, 1139, 13, 63, 90, 250, 924, 48, 1035, 2353, 3, 28, 204, 48, 976, 58, 5, 343, 166, 9, 2, 79, 33, 115, 40]


In [27]:
train_seq_lengths=[len(seq) for seq in X_train_seq]
print(pd.Series(train_seq_lengths).describe())

count    34998.000000
mean       110.681725
std         84.453840
min          3.000000
25%         58.000000
50%         82.000000
75%        135.000000
max       1342.000000
dtype: float64


In [28]:
print(pd.Series(train_seq_lengths).quantile([0.90, 0.95, 0.99]))

0.90    219.0
0.95    289.0
0.99    435.0
dtype: float64


In [37]:
MAX_LEN =250

In [38]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

In [39]:
print("X_train_pad shape:", X_train_pad.shape)
print("X_val_pad shape:", X_val_pad.shape)
print("X_test_pad shape:", X_test_pad.shape)


X_train_pad shape: (34998, 250)
X_val_pad shape: (7502, 250)
X_test_pad shape: (7500, 250)


In [40]:
print("Orginal sequence length:",len(X_train_seq[0]))
print("Padded sequence length:", len(X_train_pad[0]))
print(X_train_pad[0])

Orginal sequence length: 64
Padded sequence length: 250
[  25   12   51    1    3   21  193  184  184   64  962 1922   12   12
    3   27   13   36 1252  427   87   79 2946    1 1008   13 1062  286
   30 1768  166 2974   27    6  166 1490  674   13 1233   73 1139   13
   63   90  250  924   48 1035 2353    3   28  204   48  976   58    5
  343  166    9    2   79   33  115   40    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0 

In [ ]:
y_train =np.array(y_train)
y_val = np.array(y_val)
y_test = np.array(y_test)

print(y_train)


[1 0 0 ... 1 1 1]
[1 0 0 ... 1 1 1]


In [49]:
word_index = tokenizer.word_index
print("Vocabulary size (actual):",len(word_index))
print("First 20 items :")
print(list(word_index.items())[:20])

Vocabulary size (actual): 71761
First 20 items :
[('<OOV>', 1), ('s', 2), ('movie', 3), ('film', 4), ('t', 5), ('not', 6), ('like', 7), ('good', 8), ('time', 9), ('character', 10), ('watch', 11), ('bad', 12), ('no', 13), ('story', 14), ('see', 15), ('think', 16), ('scene', 17), ('great', 18), ('look', 19), ('know', 20)]


In [50]:
actual_vocab_size =min(VOCAB_SIZE,len(word_index)+1)
print("Actual vocab size for embedding:", actual_vocab_size)

Actual vocab size for embedding: 10000


In [ ]:
np.save("data/tokenization/X_train_pad.npy", X_train_pad)
np.save("data/tokenization/X_val_pad.npy", X_val_pad)
np.save("data/tokenization/X_test_pad.npy", X_test_pad)

np.save("data/tokenization/y_train.npy", y_train)
np.save("data/tokenization/y_val.npy", y_val)
np.save("data/tokenization/y_test.npy", y_test)

FileNotFoundError: [Errno 2] No such file or directory: 'data/tokenizer.pkl'